# 🎾 Pipeline de Auto-Labeling para Detección de Pelotas

Este notebook implementa un pipeline completo para generar datasets de detección de objetos usando:

- YOLOv8
- SAHI (Slicing Aided Hyper Inference)
- Revisión humana semi-automática
- Validación de anotaciones
- Subida optimizada a Roboflow

El flujo está diseñado para acelerar la creación de datasets de entrenamiento para visión por computador en escenarios deportivos como pádel.

---

# 🧠 Flujo General del Pipeline

El proceso completo sigue estos pasos:

1. Generar anotaciones automáticas con YOLO + SAHI
2. Detectar imágenes ambiguas con múltiples pelotas
3. Revisar manualmente las detecciones problemáticas
4. Validar la consistencia de las anotaciones
5. Subir el dataset limpio a Roboflow
6. Realizar la revisión final dentro de Roboflow

---

# ⚙️ Paso 1 — Auto Labeling con YOLO + SAHI

Este script realiza inferencia automática sobre los frames del dataset utilizando:

- YOLOv8 como detector
- SAHI para dividir imágenes grandes en slices
- Exportación automática en formato YOLO

Además:

✅ Guarda logs de imágenes procesadas  
✅ Genera imágenes debug  
✅ Detecta automáticamente casos ambiguos  
✅ Envía imágenes con múltiples detecciones a revisión manual  
✅ Genera `data.yaml` automáticamente  

El resultado queda listo para entrenamiento o subida a Roboflow.


In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Sun May 24 21:42:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   61C    P0            749W /   60W |      11MiB /   4096MiB |      8%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
"""
autolabel_sahi.py
══════════════════════════════════════════════════════
YOLO + SAHI Auto Labeler
══════════════════════════════════════════════════════

✔ Compatible con tu estructura actual:
  PROYECTOFINAL/
  ├── Assets/
  │   ├── frames_ball/
  │   └── videos/
  ├── src/
  │   ├── review_tool.py
  │   └── validate_annotations.py
  ├── best.pt
  ├── auto-labeling-final.ipynb
  └── ...

✔ Guarda processed.txt
✔ Output listo para Roboflow
✔ data.yaml automático
✔ Human-in-the-loop review
✔ Rutas dinámicas (funciona desde notebook o script)

Instalar:
pip install sahi ultralytics opencv-python pyyaml tqdm
"""

# ══════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════

import shutil
from pathlib import Path

import cv2
import yaml
from tqdm.auto import tqdm

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction


# ══════════════════════════════════════════════════════
# ROOT DEL PROYECTO
# ══════════════════════════════════════════════════════

# Funciona aunque ejecutes desde notebook o src/

try:
    ROOT_DIR = Path(__file__).resolve().parent
except:
    ROOT_DIR = Path.cwd()

# Si estás dentro de src/, subir un nivel
if ROOT_DIR.name == "src":
    ROOT_DIR = ROOT_DIR.parent


# ══════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════

MODEL_PATH = ROOT_DIR / "best.pt"

FRAMES_DIR = ROOT_DIR / "Assets" / "frames_ball"

OUTPUT_DIR = ROOT_DIR / "dataset"

CONFIDENCE = 0.20
COPY_IMAGES = True
SKIP_EMPTY = False
# ══════════════════════════════════════════════════════
# DEVICE AUTO-DETECT
# ══════════════════════════════════════════════════════

import torch

if torch.cuda.is_available():

    DEVICE = "cuda"

    DEVICE_NAME = torch.cuda.get_device_name(0)

elif torch.backends.mps.is_available():

    DEVICE = "mps"

    DEVICE_NAME = "Apple Silicon (Metal/MPS)"

else:

    DEVICE = "cpu"

    DEVICE_NAME = "CPU"

# SAHI
SLICE_HEIGHT = 640
SLICE_WIDTH = 640
OVERLAP_HEIGHT = 0.20
OVERLAP_WIDTH = 0.20

# Limitador
MAX_IMAGES = 0  # 0 para procesar todas, o un número para limitar (útil en review mode)

# Debug
SAVE_DEBUG_IMAGES = True
DEBUG_DIR = "debug"

# Review
ENABLE_REVIEW_MODE = True
REVIEW_DIR = "review"

# Procesadas
PROCESSED_LOG = "processed.txt"

IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# ══════════════════════════════════════════════════════
# UTILS
# ══════════════════════════════════════════════════════

def collect_images(folder: Path) -> list[Path]:

    imgs = [
        p for p in folder.rglob("*")
        if p.suffix.lower() in IMG_EXTENSIONS
    ]

    imgs.sort()

    return imgs


def safe_stem(img_path: Path, base_dir: Path) -> str:

    rel = img_path.relative_to(base_dir)

    parts = list(rel.parent.parts) + [rel.stem]

    return "__".join(parts)


def draw_box(img, x1, y1, x2, y2, label: str):

    color = (0, 255, 0)

    cv2.rectangle(
        img,
        (int(x1), int(y1)),
        (int(x2), int(y2)),
        color,
        2,
    )

    cv2.putText(
        img,
        label,
        (int(x1), int(y1) - 8),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        color,
        2,
    )


def load_processed_log(log_path: Path) -> set[str]:

    if not log_path.exists():
        return set()

    lines = log_path.read_text(
        encoding="utf-8"
    ).splitlines()

    return set(
        l.strip()
        for l in lines
        if l.strip()
    )


def append_to_log(log_path: Path, img_path: Path):

    with open(log_path, "a", encoding="utf-8") as f:
        f.write(str(img_path) + "\n")


# ══════════════════════════════════════════════════════
# PATHS
# ══════════════════════════════════════════════════════

frames_dir = Path(FRAMES_DIR)

output_dir = Path(OUTPUT_DIR)

images_out = output_dir / "images"
labels_out = output_dir / "labels"

debug_out = output_dir / DEBUG_DIR
review_out = output_dir / REVIEW_DIR

log_path = output_dir / PROCESSED_LOG

# Crear carpetas
images_out.mkdir(parents=True, exist_ok=True)
labels_out.mkdir(parents=True, exist_ok=True)

if SAVE_DEBUG_IMAGES:
    debug_out.mkdir(parents=True, exist_ok=True)

if ENABLE_REVIEW_MODE:
    review_out.mkdir(parents=True, exist_ok=True)


# ══════════════════════════════════════════════════════
# VALIDACIONES
# ══════════════════════════════════════════════════════

if not MODEL_PATH.exists():

    raise FileNotFoundError(
        f"No existe el modelo:\n{MODEL_PATH}"
    )

if not frames_dir.exists():

    raise FileNotFoundError(
        f"No existe frames_ball:\n{frames_dir}"
    )


# ══════════════════════════════════════════════════════
# LOG PROCESADAS
# ══════════════════════════════════════════════════════

already_processed = load_processed_log(log_path)

if already_processed:

    print(
        f"[INFO] {len(already_processed)} imágenes ya procesadas"
    )


# ══════════════════════════════════════════════════════
# IMÁGENES
# ══════════════════════════════════════════════════════

all_images = collect_images(frames_dir)

images = [
    p for p in all_images
    if str(p) not in already_processed
]

if not images:

    print("✓ Todas las imágenes ya fueron procesadas.")
    exit(0)

if MAX_IMAGES and len(images) > MAX_IMAGES:
    images = images[:MAX_IMAGES]


# ══════════════════════════════════════════════════════
# BANNER
# ══════════════════════════════════════════════════════

print(f"""
═══════════════════════════════════
YOLO + SAHI AUTO LABELER
═══════════════════════════════════

ROOT           : {ROOT_DIR}

Modelo         : {MODEL_PATH}

Frames dir     : {frames_dir}

Frames total   : {len(all_images)}
Ya procesadas  : {len(already_processed)}
A procesar     : {len(images)}

Output         : {output_dir}

Confidence     : {CONFIDENCE}
Device         : {DEVICE}
Device name    : {DEVICE_NAME}

SAHI:
  slice_height = {SLICE_HEIGHT}
  slice_width  = {SLICE_WIDTH}
  overlap_h    = {OVERLAP_HEIGHT}
  overlap_w    = {OVERLAP_WIDTH}

═══════════════════════════════════
""")


# ══════════════════════════════════════════════════════
# CARGAR MODELO
# ══════════════════════════════════════════════════════

detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=str(MODEL_PATH),
    confidence_threshold=CONFIDENCE,
    device=DEVICE,
)


# ══════════════════════════════════════════════════════
# STATS
# ══════════════════════════════════════════════════════

stats = {
    "total": 0,
    "with_boxes": 0,
    "without_boxes": 0,
    "boxes": 0,
    "review": 0,
    "skipped": len(already_processed),
}


# ══════════════════════════════════════════════════════
# LOOP PRINCIPAL
# ══════════════════════════════════════════════════════

for img_path in tqdm(images, desc="Auto-labeling"):

    stats["total"] += 1

    img = cv2.imread(str(img_path))

    if img is None:

        print(f"[WARN] No se pudo leer: {img_path}")

        append_to_log(log_path, img_path)

        continue

    h, w = img.shape[:2]

    # ──────────────────────────────────────────────
    # SAHI
    # ──────────────────────────────────────────────

    result = get_sliced_prediction(
        str(img_path),
        detection_model,
        slice_height=SLICE_HEIGHT,
        slice_width=SLICE_WIDTH,
        overlap_height_ratio=OVERLAP_HEIGHT,
        overlap_width_ratio=OVERLAP_WIDTH,
        verbose=0,
    )

    detections = []

    debug_img = img.copy()

    # ──────────────────────────────────────────────
    # DETECCIONES
    # ──────────────────────────────────────────────

    for obj in result.object_prediction_list:

        bbox = obj.bbox

        x1 = bbox.minx
        y1 = bbox.miny
        x2 = bbox.maxx
        y2 = bbox.maxy

        cls_id = int(obj.category.id)
        score = float(obj.score.value)

        xc = ((x1 + x2) / 2) / w
        yc = ((y1 + y2) / 2) / h

        bw = (x2 - x1) / w
        bh = (y2 - y1) / h

        yolo_line = (
            f"{cls_id} "
            f"{xc:.6f} "
            f"{yc:.6f} "
            f"{bw:.6f} "
            f"{bh:.6f}"
        )

        detections.append({
            "cls_id": cls_id,
            "score": score,
            "bbox": [x1, y1, x2, y2],
            "yolo": yolo_line,
        })

        stats["boxes"] += 1

    # ordenar
    detections.sort(
        key=lambda d: d["score"],
        reverse=True,
    )

    # debug draw
    for idx, det in enumerate(detections):

        x1, y1, x2, y2 = det["bbox"]

        label = f"[{idx}] {det['score']:.2f}"

        draw_box(
            debug_img,
            x1,
            y1,
            x2,
            y2,
            label,
        )

    # stats
    if detections:
        stats["with_boxes"] += 1
    else:
        stats["without_boxes"] += 1

        if SKIP_EMPTY:
            append_to_log(log_path, img_path)
            continue

    stem = safe_stem(img_path, frames_dir)

    txt_path = labels_out / f"{stem}.txt"

    # ──────────────────────────────────────────────
    # REVIEW MODE
    # ──────────────────────────────────────────────

    if ENABLE_REVIEW_MODE and len(detections) > 1:

        stats["review"] += 1

        review_img_path = review_out / f"{stem}.jpg"
        review_yaml_path = review_out / f"{stem}.yaml"

        cv2.imwrite(
            str(review_img_path),
            debug_img,
        )

        review_data = {
            "image": str(img_path),
            "label_file": str(txt_path),
            "detections": detections,
        }

        with open(review_yaml_path, "w") as f:

            yaml.dump(
                review_data,
                f,
                allow_unicode=True,
                sort_keys=False,
            )

        if COPY_IMAGES:

            shutil.copy2(
                img_path,
                images_out / f"{stem}{img_path.suffix}",
            )

        txt_path.write_text("")

        if SAVE_DEBUG_IMAGES:

            cv2.imwrite(
                str(debug_out / f"{stem}.jpg"),
                debug_img,
            )

        append_to_log(log_path, img_path)

        print(
            f"[REVIEW] {stem} → {len(detections)} detecciones"
        )

        continue

    # ──────────────────────────────────────────────
    # LABEL FINAL
    # ──────────────────────────────────────────────

    lines = [d["yolo"] for d in detections]

    txt_path.write_text(
        "\n".join(lines) + ("\n" if lines else "")
    )

    # copiar imagen
    if COPY_IMAGES:

        shutil.copy2(
            img_path,
            images_out / f"{stem}{img_path.suffix}",
        )

    # debug
    if SAVE_DEBUG_IMAGES:

        cv2.imwrite(
            str(debug_out / f"{stem}.jpg"),
            debug_img,
        )

    # log
    append_to_log(log_path, img_path)


# ══════════════════════════════════════════════════════
# DATA.YAML
# ══════════════════════════════════════════════════════

yaml_data = {
    "path": str(output_dir.resolve()),
    "train": "images",
    "val": "images",
    "nc": 1,
    "names": ["pelota"],
}

with open(output_dir / "data.yaml", "w") as f:

    yaml.dump(
        yaml_data,
        f,
        allow_unicode=True,
        sort_keys=False,
    )


# ══════════════════════════════════════════════════════
# RESUMEN
# ══════════════════════════════════════════════════════

total_procesadas = (
    stats["skipped"] + stats["total"]
)

print(f"""
╔══════════════════════════════════════╗
║          SAHI FINALIZADO ✓           ║
╠══════════════════════════════════════╣

  Procesadas      : {stats['total']}
  Con boxes       : {stats['with_boxes']}
  Sin boxes       : {stats['without_boxes']}
  Total boxes     : {stats['boxes']}
  En review       : {stats['review']}

  Total acumulado : {total_procesadas}

╠══════════════════════════════════════╣

  OUTPUT:

    {images_out}
    {labels_out}
    {debug_out}
    {review_out}

╠══════════════════════════════════════╣

  data.yaml:
    {output_dir / "data.yaml"}

╚══════════════════════════════════════╝

Listo para Roboflow 🚀
""")

[INFO] 5571 imágenes ya procesadas

═══════════════════════════════════
YOLO + SAHI AUTO LABELER
═══════════════════════════════════

ROOT           : /home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal

Modelo         : /home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/best.pt

Frames dir     : /home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/Assets/frames_ball

Frames total   : 19642
Ya procesadas  : 5571
A procesar     : 19484

Output         : /home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/dataset

Confidence     : 0.2
Device         : cuda
Device name    : NVIDIA GeForce RTX 3050 Laptop GPU

SAHI:
  slice_height = 640
  slice_width  = 640
  overlap_h    = 0.2
  overlap_w    = 0.2

═══════════════════════════════════



Auto-labeling:   0%|          | 0/19484 [00:00<?, ?it/s]

[REVIEW] Cancha_1__Partido_17__Punto_4__Cancha_1_Partido_17_Punto_4_frame_000835 → 2 detecciones
[REVIEW] Cancha_1__Partido_17__Punto_4__Cancha_1_Partido_17_Punto_4_frame_000840 → 2 detecciones
[REVIEW] Cancha_1__Partido_17__Punto_5__Cancha_1_Partido_17_Punto_5_frame_000610 → 2 detecciones
[REVIEW] Cancha_1__Partido_17__Punto_6__Cancha_1_Partido_17_Punto_6_frame_000835 → 2 detecciones
[REVIEW] Cancha_1__Partido_1__Punto_8__Cancha_1_Partido_1_Punto_8_frame_000270 → 2 detecciones
[REVIEW] Cancha_1__Partido_1__Punto_9__Cancha_1_Partido_1_Punto_9_frame_000085 → 2 detecciones
[REVIEW] Cancha_1__Partido_1__Punto_9__Cancha_1_Partido_1_Punto_9_frame_000090 → 2 detecciones
[REVIEW] Cancha_1__Partido_3__Punto_9__Cancha_1_Partido_3_Punto_9_frame_000355 → 2 detecciones
[REVIEW] Cancha_1__Partido_5__Punto_6__Cancha_1_Partido_5_Punto_6_frame_001025 → 2 detecciones
[REVIEW] Cancha_1__Partido_7__Punto_6__Cancha_1_Partido_7_Punto_6_frame_001065 → 2 detecciones
[REVIEW] Cancha_1__Partido_9__Punto_1__Can

# 🔍 Paso 2 — Enviar Casos Ambiguos a Review

Este script reescanea las anotaciones ya generadas y detecta automáticamente imágenes con múltiples detecciones.

A diferencia del autolabeler:

❌ NO vuelve a correr inferencia  
❌ NO usa YOLO nuevamente  
❌ NO recalcula bounding boxes  

Simplemente:

- Lee los `.txt`
- Detecta imágenes con más de una pelota
- Genera previews visuales
- Crea archivos YAML para revisión manual

Este paso es útil para:

- Activar review después del procesamiento
- Revisar datasets ya existentes
- Filtrar falsos positivos
- Reducir ruido en el entrenamiento


In [5]:
"""
send_to_review.py
══════════════════════════════════════════════════════
Reescanea labels YA generados y manda a review
las imágenes con múltiples detecciones.

✔ Compatible con tu estructura actual:
  PROYECTOFINAL/
  ├── Assets/
  ├── dataset/
  │   ├── images/
  │   ├── labels/
  │   ├── debug/
  │   └── review/
  ├── src/
  └── ...

✔ NO vuelve a correr YOLO
✔ NO hace inferencia otra vez
✔ SOLO analiza labels existentes

Ideal para:
- datasets ya procesados
- corregir labels malas
- activar review después
"""

# ══════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════

from pathlib import Path
import cv2
import yaml

from tqdm.auto import tqdm


# ══════════════════════════════════════════════════════
# ROOT DEL PROYECTO
# ══════════════════════════════════════════════════════

try:
    ROOT_DIR = Path(__file__).resolve().parent
except:
    ROOT_DIR = Path.cwd()

# Si ejecuta desde src/
if ROOT_DIR.name == "src":
    ROOT_DIR = ROOT_DIR.parent


# ══════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════

DATASET_DIR = ROOT_DIR / "dataset"

IMAGES_DIR = DATASET_DIR / "images"
LABELS_DIR = DATASET_DIR / "labels"

DEBUG_DIR = DATASET_DIR / "debug"

REVIEW_DIR = DATASET_DIR / "review"

REVIEW_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BOX_COLOR = (0, 255, 0)
BOX_THICKNESS = 2

CLASS_NAME = "pelota"

IMG_EXTENSIONS = [
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
]


# ══════════════════════════════════════════════════════
# VALIDACIONES
# ══════════════════════════════════════════════════════

if not DATASET_DIR.exists():

    raise FileNotFoundError(
        f"No existe dataset:\n{DATASET_DIR}"
    )

if not LABELS_DIR.exists():

    raise FileNotFoundError(
        f"No existe labels:\n{LABELS_DIR}"
    )

if not IMAGES_DIR.exists():

    raise FileNotFoundError(
        f"No existe images:\n{IMAGES_DIR}"
    )


# ══════════════════════════════════════════════════════
# UTILS
# ══════════════════════════════════════════════════════

def find_image(stem: str):

    for ext in IMG_EXTENSIONS:

        p = IMAGES_DIR / f"{stem}{ext}"

        if p.exists():
            return p

    return None


def yolo_to_xyxy(line, w, h):

    parts = line.strip().split()

    if len(parts) != 5:
        return None

    cls_id, xc, yc, bw, bh = map(float, parts)

    xc *= w
    yc *= h

    bw *= w
    bh *= h

    x1 = xc - bw / 2
    y1 = yc - bh / 2

    x2 = xc + bw / 2
    y2 = yc + bh / 2

    return {
        "cls_id": int(cls_id),
        "bbox": [x1, y1, x2, y2],
        "yolo": line.strip(),
    }


# ══════════════════════════════════════════════════════
# LABELS
# ══════════════════════════════════════════════════════

label_files = sorted(
    LABELS_DIR.glob("*.txt")
)

if not label_files:

    print("No hay labels.")
    raise SystemExit


# ══════════════════════════════════════════════════════
# BANNER
# ══════════════════════════════════════════════════════

print(f"""
═══════════════════════════════════
SEND TO REVIEW
═══════════════════════════════════

ROOT          : {ROOT_DIR}

Dataset       : {DATASET_DIR}

Images dir    : {IMAGES_DIR}
Labels dir    : {LABELS_DIR}
Review dir    : {REVIEW_DIR}

Labels total  : {len(label_files)}

═══════════════════════════════════
""")


# ══════════════════════════════════════════════════════
# STATS
# ══════════════════════════════════════════════════════

review_count = 0
missing_images = 0
invalid_labels = 0


# ══════════════════════════════════════════════════════
# LOOP
# ══════════════════════════════════════════════════════

for label_path in tqdm(
    label_files,
    desc="Generating review",
):

    lines = [
        l.strip()
        for l in label_path.read_text(
            encoding="utf-8"
        ).splitlines()
        if l.strip()
    ]

    # SOLO múltiples detecciones
    if len(lines) <= 1:
        continue

    stem = label_path.stem

    image_path = find_image(stem)

    if image_path is None:

        print(f"[WARN] Imagen no encontrada: {stem}")

        missing_images += 1

        continue

    img = cv2.imread(str(image_path))

    if img is None:

        print(f"[WARN] No se pudo leer: {image_path}")

        missing_images += 1

        continue

    h, w = img.shape[:2]

    detections = []

    # ──────────────────────────────────────────────
    # Parse YOLO
    # ──────────────────────────────────────────────

    for idx, line in enumerate(lines):

        det = yolo_to_xyxy(
            line,
            w,
            h,
        )

        if det is None:

            invalid_labels += 1
            continue

        # score fake para ordenar visualmente
        det["score"] = 1.0 - (idx * 0.01)

        detections.append(det)

    if len(detections) <= 1:
        continue

    # ──────────────────────────────────────────────
    # Debug image
    # ──────────────────────────────────────────────

    debug_img = img.copy()

    for idx, det in enumerate(detections):

        x1, y1, x2, y2 = det["bbox"]

        x1 = int(x1)
        y1 = int(y1)

        x2 = int(x2)
        y2 = int(y2)

        cv2.rectangle(
            debug_img,
            (x1, y1),
            (x2, y2),
            BOX_COLOR,
            BOX_THICKNESS,
        )

        label = f"[{idx}]"

        cv2.putText(
            debug_img,
            label,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            BOX_COLOR,
            2,
        )

    # ──────────────────────────────────────────────
    # Guardar review
    # ──────────────────────────────────────────────

    review_img_path = REVIEW_DIR / f"{stem}.jpg"

    review_yaml_path = REVIEW_DIR / f"{stem}.yaml"

    cv2.imwrite(
        str(review_img_path),
        debug_img,
    )

    review_data = {
        "image": str(image_path),
        "label_file": str(label_path),
        "detections": detections,
    }

    with open(review_yaml_path, "w") as f:

        yaml.dump(
            review_data,
            f,
            allow_unicode=True,
            sort_keys=False,
        )

    review_count += 1


# ══════════════════════════════════════════════════════
# FINAL
# ══════════════════════════════════════════════════════

print(f"""
╔══════════════════════════════════════╗
║         REVIEW GENERADO ✓            ║
╠══════════════════════════════════════╣

  Enviadas a review:
    {review_count}

  Imágenes faltantes:
    {missing_images}

  Labels inválidos:
    {invalid_labels}

╠══════════════════════════════════════╣

  Carpeta:
    {REVIEW_DIR}

╚══════════════════════════════════════╝
""")


═══════════════════════════════════
SEND TO REVIEW
═══════════════════════════════════

ROOT          : /home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal

Dataset       : /home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/dataset

Images dir    : /home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/dataset/images
Labels dir    : /home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/dataset/labels
Review dir    : /home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/dataset/review

Labels total  : 25055

═══════════════════════════════════



Generating review:   0%|          | 0/25055 [00:00<?, ?it/s]


╔══════════════════════════════════════╗
║         REVIEW GENERADO ✓            ║
╠══════════════════════════════════════╣

  Enviadas a review:
    0

  Imágenes faltantes:
    0

  Labels inválidos:
    0

╠══════════════════════════════════════╣

  Carpeta:
    /home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/dataset/review

╚══════════════════════════════════════╝



# 🧑‍💻 Paso 3 — Revisión Manual de Detecciones

En esta etapa se revisan manualmente las imágenes enviadas a `review/`.

La herramienta permite:

✅ Seleccionar la detección correcta  
✅ Eliminar falsos positivos  
✅ Navegar rápidamente entre imágenes  
✅ Guardar automáticamente las anotaciones finales  

Objetivo:

Mantener un dataset limpio y consistente antes del entrenamiento.


# ✅ Paso 4 — Validación de Anotaciones

Este script verifica que el dataset generado sea consistente.

La validación incluye:

- Labels corruptos
- Coordenadas fuera de rango
- Archivos vacíos
- Imágenes sin labels
- Labels sin imagen
- Bounding boxes inválidas

Este paso ayuda a evitar errores durante el entrenamiento del modelo.


# ☁️ Paso 5 — Subida Optimizada a Roboflow

Este script prepara automáticamente un dataset limpio para subirlo a Roboflow.

Características:

✅ Filtra únicamente imágenes válidas  
✅ Copia labels correctos  
✅ Ignora archivos innecesarios  
✅ Usa uploads paralelos para mayor velocidad  
✅ Compatible directamente con YOLO Object Detection  

La carpeta `upload/` se genera automáticamente antes de subir el dataset.


In [4]:
"""
fast_roboflow_upload.py
══════════════════════════════════════════════════════
Prepara un dataset limpio y lo sube rápido a Roboflow
usando upload_dataset() + workers paralelos.

✔ Compatible con tu estructura actual:
  PROYECTOFINAL/
  ├── dataset/
  │   ├── images/
  │   ├── labels/
  │   ├── data.yaml
  │   ├── debug/
  │   └── review/
  ├── src/
  ├── best.pt
  └── ...

✔ Crea automáticamente:
    upload/
    ├── images/
    ├── labels/
    └── data.yaml

✔ SOLO sube dataset limpio
✔ Ignora debug/review
✔ Validación automática
✔ Compatible con notebooks y scripts
"""

# ══════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════

import shutil
from pathlib import Path

from roboflow import Roboflow
from tqdm.auto import tqdm

from dotenv import load_dotenv
import os

# ══════════════════════════════════════════════════════
# ROOT DEL PROYECTO
# ══════════════════════════════════════════════════════

try:
    ROOT_DIR = Path(__file__).resolve().parent
except:
    ROOT_DIR = Path.cwd()

# Si ejecuta desde src/
if ROOT_DIR.name == "src":
    ROOT_DIR = ROOT_DIR.parent


# ══════════════════════════════════════════════════════
# CARGAR .ENV
# ══════════════════════════════════════════════════════

load_dotenv()


# ══════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════

API_KEY = os.getenv("ROBOFLOW_API_KEY")

WORKSPACE = "santiagos-workspace-c6qdy"
PROJECT = "trackball_padelanalytics1"

SOURCE_DATASET = ROOT_DIR / "dataset"

UPLOAD_DIR = ROOT_DIR / "upload"

NUM_WORKERS = 20
NUM_RETRIES = 1

PROJECT_TYPE = "object-detection"

IMG_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


# ══════════════════════════════════════════════════════
# VALIDACIONES
# ══════════════════════════════════════════════════════

if not SOURCE_DATASET.exists():

    raise FileNotFoundError(
        f"No existe dataset:\n{SOURCE_DATASET}"
    )

if not API_KEY:

    raise ValueError(
        "Debes colocar tu API_KEY de Roboflow."
    )


# ══════════════════════════════════════════════════════
# RESET UPLOAD DIR
# ══════════════════════════════════════════════════════

def reset_upload_dir(upload_dir: Path):

    if upload_dir.exists():
        shutil.rmtree(upload_dir)

    (upload_dir / "images").mkdir(
        parents=True,
        exist_ok=True,
    )

    (upload_dir / "labels").mkdir(
        parents=True,
        exist_ok=True,
    )


# ══════════════════════════════════════════════════════
# PREPARE DATASET
# ══════════════════════════════════════════════════════

def prepare_dataset(source: Path, upload: Path):

    images_src = source / "images"
    labels_src = source / "labels"

    yaml_src = source / "data.yaml"

    if not images_src.exists():

        raise FileNotFoundError(
            f"No existe:\n{images_src}"
        )

    if not labels_src.exists():

        raise FileNotFoundError(
            f"No existe:\n{labels_src}"
        )

    if not yaml_src.exists():

        raise FileNotFoundError(
            f"No existe:\n{yaml_src}"
        )

    image_files = []

    for file in images_src.iterdir():

        if file.suffix.lower() in IMG_EXTENSIONS:
            image_files.append(file)

    print(f"\n🖼️  Imágenes encontradas: {len(image_files)}")

    copied = 0
    missing = 0
    empty_labels = 0

    for img_path in tqdm(
        image_files,
        desc="Preparando dataset",
    ):

        label_path = labels_src / f"{img_path.stem}.txt"

        # no existe label
        if not label_path.exists():

            missing += 1
            continue

        # label vacío
        label_content = label_path.read_text(
            encoding="utf-8"
        ).strip()

        if not label_content:

            empty_labels += 1
            continue

        shutil.copy2(
            img_path,
            upload / "images" / img_path.name
        )

        shutil.copy2(
            label_path,
            upload / "labels" / label_path.name
        )

        copied += 1

    # copiar data.yaml
    shutil.copy2(
        yaml_src,
        upload / "data.yaml"
    )

    return {
        "copied": copied,
        "missing": missing,
        "empty": empty_labels,
    }


# ══════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════

def main():

    source = Path(SOURCE_DATASET)
    upload = Path(UPLOAD_DIR)

    print(f"""
╔══════════════════════════════════════════╗
║      FAST ROBOFLOW UPLOADER 🚀          ║
╚══════════════════════════════════════════╝

ROOT:
{ROOT_DIR}

Dataset:
{source}

Upload dir:
{upload}
""")

    # ──────────────────────────────────────────────
    # Limpiar upload
    # ──────────────────────────────────────────────

    print("🧹 Limpiando carpeta upload...")

    reset_upload_dir(upload)

    # ──────────────────────────────────────────────
    # Preparar dataset
    # ──────────────────────────────────────────────

    print("📦 Preparando dataset limpio...")

    stats = prepare_dataset(
        source,
        upload,
    )

    print(f"""
══════════════════════════════════════════
✅ Dataset preparado

Imágenes copiadas : {stats['copied']}
Sin label          : {stats['missing']}
Labels vacíos      : {stats['empty']}

Carpeta final:
{upload.resolve()}
══════════════════════════════════════════
""")

    # ──────────────────────────────────────────────
    # Conectar Roboflow
    # ──────────────────────────────────────────────

    print("☁️  Conectando a Roboflow...")

    rf = Roboflow(
        api_key=API_KEY
    )

    workspace = rf.workspace(WORKSPACE)

    print("✅ Conectado\n")

    # ──────────────────────────────────────────────
    # Upload
    # ──────────────────────────────────────────────

    print("🚀 Subiendo dataset...\n")

    workspace.upload_dataset(
        str(upload),
        PROJECT,
        num_workers=NUM_WORKERS,
        project_type=PROJECT_TYPE,
        num_retries=NUM_RETRIES,
    )

    # ──────────────────────────────────────────────
    # FINAL
    # ──────────────────────────────────────────────

    print(f"""
╔══════════════════════════════════════════╗
║           UPLOAD COMPLETADO ✅           ║
╠══════════════════════════════════════════╣

  Workspace : {WORKSPACE}
  Project   : {PROJECT}

  Workers   : {NUM_WORKERS}
  Retries   : {NUM_RETRIES}

╠══════════════════════════════════════════╣

  Dataset:
  {source}

  Upload:
  {upload}

╠══════════════════════════════════════════╣

  Proyecto:
  https://app.roboflow.com/{WORKSPACE}/{PROJECT}

╚══════════════════════════════════════════╝
""")


# ══════════════════════════════════════════════════════
# ENTRYPOINT
# ══════════════════════════════════════════════════════

if __name__ == "__main__":
    main()


╔══════════════════════════════════════════╗
║      FAST ROBOFLOW UPLOADER 🚀          ║
╚══════════════════════════════════════════╝

ROOT:
/home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal

Dataset:
/home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/dataset

Upload dir:
/home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/upload

🧹 Limpiando carpeta upload...
📦 Preparando dataset limpio...

🖼️  Imágenes encontradas: 25055


Preparando dataset:   0%|          | 0/25055 [00:00<?, ?it/s]


══════════════════════════════════════════
✅ Dataset preparado

Imágenes copiadas : 4560
Sin label          : 0
Labels vacíos      : 20495

Carpeta final:
/home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/upload
══════════════════════════════════════════

☁️  Conectando a Roboflow...
loading Roboflow workspace...
✅ Conectado

🚀 Subiendo dataset...

loading Roboflow project...
Uploading to existing project santiagos-workspace-c6qdy/trackball_padelanalytics1
[UPLOADED] /home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/upload/images/Cancha_1_Partido_5_Punto_5_frame_001435.jpg (ZYsfxLyIUiUp0xCdxsRJ) [1.6s] / annotations = OK [0.5s]
[UPLOADED] /home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/upload/images/Cancha_1_Partido_5_Punto_5_frame_001425.jpg (duvtRE5sjOlFUbmQwDMa) [1.9s] / annotations = OK [0.3s]
[UPLOADED] /home/lemarsito/Documentos/2026 - 1/ANALITICA/proyectofinal/upload/images/Cancha_1_Partido_5_Punto_5_frame_000920.jpg (IhIGrtmcBdDVJZGGHDHk) [2.0s]

# 🧪 Paso 6 — Revisión Final en Roboflow

Finalmente, el dataset se revisa directamente desde Roboflow para:

- Corregir anotaciones restantes
- Revisar imágenes difíciles
- Verificar consistencia general
- Generar versiones del dataset
- Entrenar modelos finales

Una vez validado el dataset, queda listo para entrenamiento y exportación.

---

# 🚀 Resultado Final

Al finalizar el pipeline se obtiene:

✅ Dataset limpio  
✅ Anotaciones validadas  
✅ Casos ambiguos revisados  
✅ Compatible con YOLOv8  
✅ Listo para entrenamiento y exportación  

Estructura final esperada:

```text
dataset/
├── images/
├── labels/
├── debug/
├── review/
├── processed.txt
└── data.yaml